# 02 — Detailed Analysis

This notebook covers:
- Bivariate analysis (numeric and categorical features vs. TARGET)
- Correlation matrix and highly-correlated feature pairs
- Statistical significance tests (chi-squared, point-biserial)
- Feature engineering (CREDIT_TO_INCOME, AGE_YEARS, etc.)

**Prerequisites:** Run `01_data_exploration.ipynb` first, or re-run the data loading cell below.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import (
    DATA_RAW_PATH, APPLICATION_DATA_FILE,
    KEY_NUMERIC_COLUMNS, KEY_CATEGORICAL_COLUMNS,
    CORRELATION_THRESHOLD, FIGURE_SIZE,
)
from src.data_loader import load_application_data, clean_data
from src.analysis import (
    calculate_default_statistics,
    compute_correlation_matrix,
    engineer_features,
)
from src.visualizations import (
    plot_default_by_income,
    plot_correlation_heatmap,
    plot_age_vs_default,
    plot_credit_to_income,
)
from src.eda_utils import get_categorical_default_rates

%matplotlib inline
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load & Clean Data

In [ ]:
data_path = os.path.join(DATA_RAW_PATH, APPLICATION_DATA_FILE)

try:
    df_raw = load_application_data(data_path)
except FileNotFoundError:
    np.random.seed(42)
    n = 2000
    df_raw = pd.DataFrame({
        'TARGET': np.random.choice([0, 1], size=n, p=[0.92, 0.08]),
        'AMT_INCOME_TOTAL': np.random.lognormal(11, 0.5, n),
        'AMT_CREDIT': np.random.lognormal(12, 0.5, n),
        'AMT_ANNUITY': np.random.lognormal(9, 0.4, n),
        'AMT_GOODS_PRICE': np.random.lognormal(12, 0.5, n),
        'DAYS_BIRTH': -np.random.randint(7000, 25000, n),
        'DAYS_EMPLOYED': np.where(
            np.random.random(n) < 0.1, 365243,
            -np.random.randint(1, 10000, n)
        ),
        'CNT_CHILDREN': np.random.randint(0, 5, n),
        'CNT_FAM_MEMBERS': np.random.randint(1, 6, n),
        'CODE_GENDER': np.random.choice(['M', 'F'], n),
        'NAME_INCOME_TYPE': np.random.choice(
            ['Working', 'Commercial associate', 'Pensioner', 'State servant',
             'Unemployed', 'Maternity leave'],
            n, p=[0.48, 0.20, 0.18, 0.08, 0.03, 0.03]
        ),
        'NAME_EDUCATION_TYPE': np.random.choice(
            ['Secondary / secondary special', 'Higher education',
             'Incomplete higher', 'Lower secondary'],
            n, p=[0.7, 0.2, 0.07, 0.03]
        ),
        'OCCUPATION_TYPE': np.where(
            np.random.random(n) < 0.3, np.nan,
            np.random.choice(['Laborers', 'Core staff', 'Sales staff', 'Managers'], n)
        ),
    })

df = clean_data(df_raw)
print(f'Working dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2. Default Statistics

In [ ]:
stats_summary = calculate_default_statistics(df)
for k, v in stats_summary.items():
    if isinstance(v, float):
        print(f'{k}: {v:.4f}')
    else:
        print(f'{k}: {v}')

## 3. Feature Engineering

In [ ]:
df = engineer_features(df)
engineered = [c for c in ['CREDIT_TO_INCOME', 'ANNUITY_TO_INCOME',
                           'INCOME_PER_PERSON', 'AGE_YEARS', 'EMPLOYMENT_YEARS']
              if c in df.columns]
print('Engineered features added:', engineered)
df[engineered].describe().round(2)

## 4. Bivariate Analysis — Default by Income Type

In [ ]:
if 'NAME_INCOME_TYPE' in df.columns:
    fig = plot_default_by_income(df)
    plt.tight_layout()
    plt.show()

## 5. Age vs. Default Rate

In [ ]:
if 'DAYS_BIRTH' in df.columns or 'AGE_YEARS' in df.columns:
    fig = plot_age_vs_default(df)
    plt.tight_layout()
    plt.show()

## 6. Credit-to-Income Distribution

In [ ]:
if 'CREDIT_TO_INCOME' in df.columns:
    fig = plot_credit_to_income(df)
    plt.tight_layout()
    plt.show()

## 7. Correlation Matrix

In [ ]:
numeric_cols = [c for c in KEY_NUMERIC_COLUMNS + ['CREDIT_TO_INCOME', 'AGE_YEARS']
                if c in df.columns]
if len(numeric_cols) >= 2:
    corr_matrix, high_corr_pairs = compute_correlation_matrix(
        df, columns=numeric_cols, threshold=CORRELATION_THRESHOLD
    )
    print(f'\nHighly correlated pairs (|r| > {CORRELATION_THRESHOLD}):')
    for feat1, feat2, r in high_corr_pairs:
        print(f'  {feat1} ↔ {feat2}: r = {r:.3f}')

    fig = plot_correlation_heatmap(df, columns=numeric_cols)
    plt.tight_layout()
    plt.show()

## 8. Statistical Significance Tests

In [ ]:
# Point-biserial correlation for numeric features vs TARGET
results = []
for col in numeric_cols:
    if col == 'TARGET':
        continue
    subset = df[[col, 'TARGET']].dropna()
    if len(subset) > 30:
        r, p = stats.pointbiserialr(subset['TARGET'], subset[col])
        results.append({'feature': col, 'r': round(r, 4), 'p_value': round(p, 6)})

results_df = pd.DataFrame(results).sort_values('r', key=abs, ascending=False)
print('Point-biserial correlation with TARGET:')
results_df

In [ ]:
# Chi-squared tests for categorical features vs TARGET
cat_cols = [c for c in KEY_CATEGORICAL_COLUMNS if c in df.columns]
chi2_results = []
for col in cat_cols:
    contingency = pd.crosstab(df[col], df['TARGET'])
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    # Cramér's V effect size
    n = contingency.sum().sum()
    cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
    chi2_results.append({'feature': col, 'chi2': round(chi2, 2),
                         'p_value': round(p, 6), "cramers_v": round(cramers_v, 4)})

chi2_df = pd.DataFrame(chi2_results).sort_values('cramers_v', ascending=False)
print('Chi-squared tests vs TARGET (sorted by effect size):')
chi2_df

---
**Next:** Proceed to `03_insights_recommendations.ipynb` for findings summary and business recommendations.